### Importing libraries

In [8]:
# Installing Alchemy and PyMySQL
#%pip install sqlalchemy
#%pip install pymysql
#%pip install lat-lon-parser
#%pip install python-dotenv


import pandas as pd
import requests
import os
import json
from datetime import datetime
from bs4 import BeautifulSoup
from lat_lon_parser import parse
from sqlalchemy import create_engine

### Scraping data from Wikipedia

In [33]:
# Scraping data from Wikipedia
# Sources
urls = [
    "https://en.wikipedia.org/wiki/Berlin",
    "https://en.wikipedia.org/wiki/Hamburg",
    "https://en.wikipedia.org/wiki/Munich"
]

# Storage lists
cities = []
countries = []
latitudes = []
longitudes = []
population = []

# Loop to retrieve values
for url in urls:

    response = requests.get(url, headers={'User-Agent': 'Chrome/134.0.0.0'})
    soup = BeautifulSoup(response.content, 'html.parser')

    # CITY NAME
    city = soup.find("h1").get_text()

    # COUNTRY
    box = soup.find("table", class_="infobox")
    country = None

    for row in box.find_all("tr"):
        header = row.find("th")
        value = row.find("td")

        if header and value:
            if "country" in header.get_text().lower():
                country = value.get_text(" ", strip=True)
                break

    # COORDINATES
    geo = soup.find("span", class_="geo")

    latitude = None
    longitude = None

    city_latitude = soup.find(class_="latitude").get_text()
    city_longitude = soup.find(class_="longitude").get_text()

    latitude = parse(city_latitude)
    longitude = parse(city_longitude)

    # POPULATION

    population_text = soup.find(string="Population").find_next("td").get_text()
    population_clean = int(population_text.replace(",", ""))
    today = datetime.today().strftime("%d.%m.%Y")

    # STORE RESULTS
    cities.append(city)
    countries.append(country)
    latitudes.append(latitude)
    longitudes.append(longitude)
    population.append(population_clean)

    # Data Frame
    cities_df = pd.DataFrame({
    "city": cities,
    "country": countries,
    "latitude": latitudes,
    "longitude": longitudes,
    "population": population,
    "timestamp_population": today
})

# Rounding coordinates
cities_df["latitude"] = cities_df["latitude"].astype(float).round(4)
cities_df["longitude"] = cities_df["longitude"].astype(float).round(4)

print(cities_df)

      city  country  latitude  longitude  population timestamp_population
0   Berlin  Germany   52.5200     13.405     3596999           26.05.2026
1  Hamburg  Germany   53.5500     10.000     1973896           26.05.2026
2   Munich  Germany   48.1375     11.575     1505005           26.05.2026


In [55]:
# Connecting Python to SQL with SQLAlchemy and PyMySQL 
schema = "cities_db"
host = "127.0.0.1"
user = "root"
password = os.getenv("MYSQL_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
engine = create_engine(connection_string)

In [35]:
# Start with city table
city_table = cities_df[["city"]].copy()

In [36]:
# Storing data to SQL
city_table[["city"]].to_sql(
    "cities",
    con=engine,
    if_exists="append",
    index=False
)
cities_from_sql = pd.read_sql("SELECT * FROM cities", con=engine) #read IDs from SQL

# Merge back ids
merged = cities_df.merge(
    cities_from_sql,
    on="city",
    how="left"
)

# Add population, coordinates and country
population_table = merged[["city_id", "population"]].copy()
coordinates_table = merged[["city_id", "latitude", "longitude"]].copy()
country_table = merged[["city_id", "country"]].copy()

In [37]:
print(os.listdir())

['0_sql_recapitulation_challenges.sql', '1_sql_maths_challenges.sql', '1_webScraping_Challenge2.ipynb', '1_webScraping__structure.ipynb', '2_pyhon_to_sql_cities_schema.sql', '2_python_to_sql.ipynb', '2_python_to_sql_cities_schema.ipynb', '2_python_to_sql_weather.sql', '2_sql_subquery_cte_temp_table_challenges.sql', '2_sql_subquery_cte_temp_table_codealong.sql', '2_subquery_cte_temp_table_solutions.sql', '2_web_scraping_sample_solution.ipynb', '3_sql_table_creation.sql', '3_sql_window_functions_challenges.sql', '3_sql_window_functions_codealong.sql', '3_sql_window_functions_solutions.sql', '4_intro_to_apis.ipynb', '4_sql_stored_views_procedures_functions_challenges.sql', '4_sql_stored_views_procedures_functions_codealong.sql', '4_sql_stored_views_procedures_functions_solutions.sql', '5_aeroDataBox_introduction.ipynb', 'python_to_sql_solutions.ipynb']


In [38]:
# Send remaining tables to SQL
population_table.to_sql("population", con=engine, if_exists="append", index=False)
coordinates_table.to_sql("coordinates", con=engine, if_exists="append", index=False)
country_table.to_sql("country", con=engine, if_exists="append", index=False)

3

In [39]:
pd.read_sql("""
SELECT c.city,
       p.population,
       co.latitude,
       co.longitude,
       ctr.country
FROM cities c
JOIN population p USING(city_id)
JOIN coordinates co USING(city_id)
JOIN country ctr USING(city_id)
""", con=engine)

,city,population,latitude,longitude,country
0,Berlin,3596999,52.5200,13.405,Germany
1,Hamburg,1973896,53.5500,10.000,Germany
2,Munich,1505005,48.1375,11.575,Germany


### Scraping weather data with APIs

In [ ]:
# Scraping weather data using APIs

# Defining the key
api_key = os.getenv("OPENWEATHER_KEY")

In [41]:
# Loop to retrieve weather data for each city
def weather_dataframe(cities_df, api_key):
    weather_data = []

    for _, row in cities_df.iterrows():
        city = row["city"]
        api_lat = row["latitude"]
        api_lon = row["longitude"]

        url = (
            f"https://api.openweathermap.org/data/2.5/forecast"
            f"?lat={api_lat}&lon={api_lon}&appid={api_key}&units=metric"
        )

        response = requests.get(url)
        if response.status_code != 200:
            print(f"Error for {city}: {response.status_code}")
            continue

        data = response.json()

        for item in data["list"]:
            weather_data.append({
                "city": city,
                "datetime": item["dt_txt"],
                "temp": item["main"]["temp"],
                "feels_like": item["main"]["feels_like"],
                "humidity": item["main"]["humidity"],
                "weather": item["weather"][0]["main"],
                "wind_speed": item["wind"]["speed"],
                "pop": item.get("pop", None),
                "rain_3h": item.get("rain", {}).get("3h", 0),
                "snow_3h": item.get("snow", {}).get("3h", 0),
                "retrieved_at": datetime.now()
            })

    return pd.DataFrame(weather_data)

In [42]:
# Use and inspect weather dataframe
weather_df = weather_dataframe(cities_df, api_key)

weather_df

,city,datetime,temp,feels_like,humidity,weather,wind_speed,pop,rain_3h,snow_3h,retrieved_at
0,Berlin,2026-05-26 18:00:00,28.83,28.71,43,Clear,3.88,0.00,0.00,0,2026-05-26 17:25:29.433811
1,Berlin,2026-05-26 21:00:00,22.89,22.83,61,Clear,3.80,0.00,0.00,0,2026-05-26 17:25:29.433819
2,Berlin,2026-05-27 00:00:00,14.83,14.48,81,Clear,3.78,0.00,0.00,0,2026-05-26 17:25:29.433822
3,Berlin,2026-05-27 03:00:00,12.88,12.50,87,Clear,3.56,0.00,0.00,0,2026-05-26 17:25:29.433824
4,Berlin,2026-05-27 06:00:00,16.81,15.77,47,Clear,5.05,0.00,0.00,0,2026-05-26 17:25:29.433826
...,...,...,...,...,...,...,...,...,...,...,...
115,Munich,2026-05-31 03:00:00,13.92,13.72,90,Rain,2.94,0.31,0.27,0,2026-05-26 17:25:30.379350
116,Munich,2026-05-31 06:00:00,17.64,17.31,71,Clouds,1.68,0.00,0.00,0,2026-05-26 17:25:30.379352
117,Munich,2026-05-31 09:00:00,26.13,26.13,40,Clouds,1.15,0.00,0.00,0,2026-05-26 17:25:30.379354
118,Munich,2026-05-31 12:00:00,28.62,27.68,32,Clouds,1.91,0.00,0.00,0,2026-05-26 17:25:30.379355


In [43]:
# Merge with cities info
weather_full = weather_df.merge(
    cities_df,
    on="city",
    how="left"
)

weather_full

,city,datetime,temp,feels_like,humidity,weather,wind_speed,pop,rain_3h,snow_3h,retrieved_at,country,latitude,longitude,population,timestamp_population
0,Berlin,2026-05-26 18:00:00,28.83,28.71,43,Clear,3.88,0.00,0.00,0,2026-05-26 17:25:29.433811,Germany,52.5200,13.405,3596999,26.05.2026
1,Berlin,2026-05-26 21:00:00,22.89,22.83,61,Clear,3.80,0.00,0.00,0,2026-05-26 17:25:29.433819,Germany,52.5200,13.405,3596999,26.05.2026
2,Berlin,2026-05-27 00:00:00,14.83,14.48,81,Clear,3.78,0.00,0.00,0,2026-05-26 17:25:29.433822,Germany,52.5200,13.405,3596999,26.05.2026
3,Berlin,2026-05-27 03:00:00,12.88,12.50,87,Clear,3.56,0.00,0.00,0,2026-05-26 17:25:29.433824,Germany,52.5200,13.405,3596999,26.05.2026
4,Berlin,2026-05-27 06:00:00,16.81,15.77,47,Clear,5.05,0.00,0.00,0,2026-05-26 17:25:29.433826,Germany,52.5200,13.405,3596999,26.05.2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,Munich,2026-05-31 03:00:00,13.92,13.72,90,Rain,2.94,0.31,0.27,0,2026-05-26 17:25:30.379350,Germany,48.1375,11.575,1505005,26.05.2026
116,Munich,2026-05-31 06:00:00,17.64,17.31,71,Clouds,1.68,0.00,0.00,0,2026-05-26 17:25:30.379352,Germany,48.1375,11.575,1505005,26.05.2026
117,Munich,2026-05-31 09:00:00,26.13,26.13,40,Clouds,1.15,0.00,0.00,0,2026-05-26 17:25:30.379354,Germany,48.1375,11.575,1505005,26.05.2026
118,Munich,2026-05-31 12:00:00,28.62,27.68,32,Clouds,1.91,0.00,0.00,0,2026-05-26 17:25:30.379355,Germany,48.1375,11.575,1505005,26.05.2026


In [44]:
# Merge with cities id info from SQL
cities_in_db = pd.read_sql("SELECT city_id, city FROM cities", con=connection_string)

weather_df = weather_df.merge(cities_in_db, on="city", how="left")

In [45]:
# Only keep relevant rows
final_weather_df = weather_df[[
    "city_id", "datetime", "temp", "feels_like",
    "humidity", "weather", "wind_speed", "pop", "rain_3h", "snow_3h", "retrieved_at"
]]

In [46]:
# Store weather data to SQL
final_weather_df.to_sql(
    "weather",
    con=connection_string,
    if_exists="append",
    index=False
)


120

In [47]:
pd.read_sql("SELECT * FROM weather", con=engine)

,id,city_id,datetime,temp,feels_like,humidity,weather,wind_speed,pop,rain_3h,snow_3h,retrieved_at
0,1,1,2026-05-26 18:00:00,28.83,28.71,43,Clear,3.88,0.00,0.00,0.0,2026-05-26 17:25:29
1,2,1,2026-05-26 21:00:00,22.89,22.83,61,Clear,3.80,0.00,0.00,0.0,2026-05-26 17:25:29
2,3,1,2026-05-27 00:00:00,14.83,14.48,81,Clear,3.78,0.00,0.00,0.0,2026-05-26 17:25:29
3,4,1,2026-05-27 03:00:00,12.88,12.50,87,Clear,3.56,0.00,0.00,0.0,2026-05-26 17:25:29
4,5,1,2026-05-27 06:00:00,16.81,15.77,47,Clear,5.05,0.00,0.00,0.0,2026-05-26 17:25:29
...,...,...,...,...,...,...,...,...,...,...,...,...
115,116,3,2026-05-31 03:00:00,13.92,13.72,90,Rain,2.94,0.31,0.27,0.0,2026-05-26 17:25:30
116,117,3,2026-05-31 06:00:00,17.64,17.31,71,Clouds,1.68,0.00,0.00,0.0,2026-05-26 17:25:30
117,118,3,2026-05-31 09:00:00,26.13,26.13,40,Clouds,1.15,0.00,0.00,0.0,2026-05-26 17:25:30
118,119,3,2026-05-31 12:00:00,28.62,27.68,32,Clouds,1.91,0.00,0.00,0.0,2026-05-26 17:25:30


### Getting airports data from AeroDataBox

In [ ]:
# Request code
import requests

url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

querystring = {"lat":"52.52","lon":"13.405","radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

headers = {
	"x-rapidapi-key":  os.getenv("RAPIDAPI_KEY"),
	"x-rapidapi-host": "aerodatabox.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

In [ ]:
# Loop to retrieve airport data for each city
def get_airports(latitudes, longitudes):
    # API headers
    headers = {
        "x-rapidapi-key":  os.getenv("RAPID_API_KEY"),
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    # DataFrame to store results
    all_airports = []

    for lat, lon in zip(latitudes, longitudes):
    # Construct the URL with the latitude and longitude
        url = url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        querystring = {"lat":lat,"lon":lon,"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

        # Make the API request
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            all_airports.append(airports)

    return pd.concat(all_airports, ignore_index=True)

### Getting flights data fron AeroDataBox

In [ ]:
# Requesting data fro BER from Aerodatabox API
import requests

url = "https://aerodatabox.p.rapidapi.com/flights/airports/iata/BER/2026-05-29T12:00/2026-05-29T23:59"

querystring = {"withLeg":"true","direction":"Both","withCancelled":"false","withCodeshared":"false","withCargo":"false","withPrivate":"false","withLocation":"false"}

headers = {
	"x-rapidapi-key": os.getenv("RAPID_API_KEY"),
	"x-rapidapi-host": "aerodatabox.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

{'departures': [{'departure': {'scheduledTime': {'utc': '2026-05-29 10:00Z', 'local': '2026-05-29 12:00+02:00'}, 'revisedTime': {'utc': '2026-05-29 10:00Z', 'local': '2026-05-29 12:00+02:00'}, 'terminal': '1', 'checkInDesk': '021-023', 'quality': ['Basic', 'Live']}, 'arrival': {'airport': {'icao': 'DTMB', 'iata': 'MIR', 'name': 'Monastir', 'countryCode': 'tn', 'timeZone': 'Africa/Tunis'}, 'scheduledTime': {'utc': '2026-05-29 12:50Z', 'local': '2026-05-29 13:50+01:00'}, 'revisedTime': {'utc': '2026-05-29 12:50Z', 'local': '2026-05-29 13:50+01:00'}, 'quality': ['Basic', 'Live']}, 'number': 'BJ 249', 'status': 'Expected', 'codeshareStatus': 'IsOperator', 'isCargo': False, 'aircraft': {'reg': 'TS-INC', 'model': 'Airbus A320-200'}, 'airline': {'name': 'Nouvelair Tunisie', 'iata': 'BJ', 'icao': 'LBT'}}, {'departure': {'scheduledTime': {'utc': '2026-05-29 10:05Z', 'local': '2026-05-29 12:05+02:00'}, 'revisedTime': {'utc': '2026-05-29 10:05Z', 'local': '2026-05-29 12:05+02:00'}, 'terminal': '2

In [30]:
# Check key structure of response
data = response.json()

print(data.keys())

dict_keys(['departures', 'arrivals'])


In [31]:
# Turn response into dataframe
pd.json_normalize(response.json()['arrivals'])

,number,status,codeshareStatus,isCargo,departure.airport.icao,departure.airport.iata,departure.airport.name,departure.airport.countryCode,departure.airport.timeZone,departure.scheduledTime.utc,...,arrival.quality,aircraft.reg,aircraft.modeS,aircraft.model,airline.name,airline.iata,airline.icao,departure.checkInDesk,departure.gate,callSign
0,LH 1926,Expected,IsOperator,False,EDDM,MUC,Munich,de,Europe/Berlin,2026-05-29 09:00Z,...,"[Basic, Live]",D-AIDN,3C648E,Airbus A321,Lufthansa,LH,DLH,NaN,NaN,NaN
1,U2 5004,Expected,IsOperator,False,LGKR,CFU,Kerkyra Island,gr,Europe/Athens,2026-05-29 07:35Z,...,"[Basic, Live]",OE-ISC,440614,Airbus A321 NEO,easyJet,U2,EZY,19-21,09,NaN
2,FR 227,Expected,IsOperator,False,LEPA,PMI,Palma De Mallorca,es,Europe/Madrid,2026-05-29 07:25Z,...,"[Basic, Live]",EI-HET,4CAD29,Boeing 737-800,Ryanair,FR,RYR,203-218,NaN,NaN
3,FR 5475,Expected,IsOperator,False,GMMX,RAK,Marrakech,ma,Africa/Casablanca,2026-05-29 06:10Z,...,"[Basic, Live]",EI-EMA,4CA849,Boeing 737-800,Ryanair,FR,RYR,NaN,NaN,NaN
4,TP 530,Expected,IsOperator,False,LPPT,LIS,Lisbon,pt,Europe/Lisbon,2026-05-29 06:50Z,...,"[Basic, Live]",NaN,NaN,Airbus A320-200 (sharklets),TAP Air Portugal,TP,TAP,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196,FR 5418,Expected,IsOperator,False,EIDW,DUB,Dublin,ie,Europe/Dublin,2026-05-29 18:45Z,...,"[Basic, Live]",EI-EKN,4CA80E,Boeing 737-800,Ryanair,FR,RYR,1204-1319,NaN,NaN
197,FR 3361,Expected,IsOperator,False,LICJ,PMO,Palermo,it,Europe/Rome,2026-05-29 18:30Z,...,"[Basic, Live]",EI-DWD,4CA569,Boeing 737-800,Ryanair,FR,RYR,NaN,NaN,NaN
198,EW 8671,Expected,IsOperator,False,LGKO,KGS,Kos Island,gr,Europe/Athens,2026-05-29 18:00Z,...,"[Basic, Live]",D-AEED,3C54A4,Airbus A321 NEO,Eurowings,EW,EWG,04-06,NaN,NaN
199,SR 7359,Expected,IsOperator,False,LBBG,BOJ,Burgas,bg,Europe/Sofia,2026-05-29 18:30Z,...,"[Basic, Live]",NaN,NaN,Airbus A320,SundAir,SR,SDR,NaN,NaN,NaN


In [ ]:
# Keep and append relevant data in df
arrivals_list = []

for flight in data["arrivals"]:

    arrivals_list.append({
        "flight_number": flight.get("number"),
        "airline": flight.get("airline", {}).get("name"),
        "departure_airport": flight.get("departure", {})
                                   .get("airport", {})
                                   .get("name"),

        "departure_iata": flight.get("departure", {})
                                .get("airport", {})
                                .get("iata"),

        "arrival_time_local": flight.get("arrival", {})
                                    .get("scheduledTime", {})
                                    .get("local"),

        "status": flight.get("status"),

        "aircraft": flight.get("aircraft", {})
                          .get("model")
    })

arrivals_df = pd.DataFrame(arrivals_list)

arrivals_df.head()

,flight_number,airline,departure_airport,departure_iata,arrival_time_local,status,aircraft
0,LH 1926,Lufthansa,Munich,MUC,2026-05-29 12:05+02:00,Expected,Airbus A321-200
1,U2 5004,easyJet,Kerkyra Island,CFU,2026-05-29 12:05+02:00,Expected,Airbus A321 NEO
2,FR 227,Ryanair,Palma De Mallorca,PMI,2026-05-29 12:10+02:00,Expected,Boeing 737 MAX 8
3,FR 5475,Ryanair,Marrakech,RAK,2026-05-29 12:20+02:00,Expected,Boeing 737-800 (winglets)
4,TP 530,TAP Air Portugal,Lisbon,LIS,2026-05-29 12:20+02:00,Expected,Airbus A320-200 (sharklets)


In [50]:
arrivals_df["airport"] = "BER"
arrivals_df

,flight_number,airline,departure_airport,departure_iata,arrival_time_local,status,aircraft,airport
0,LH 1926,Lufthansa,Munich,MUC,2026-05-29 12:05+02:00,Expected,Airbus A321-200,BER
1,U2 5004,easyJet,Kerkyra Island,CFU,2026-05-29 12:05+02:00,Expected,Airbus A321 NEO,BER
2,FR 227,Ryanair,Palma De Mallorca,PMI,2026-05-29 12:10+02:00,Expected,Boeing 737 MAX 8,BER
3,FR 5475,Ryanair,Marrakech,RAK,2026-05-29 12:20+02:00,Expected,Boeing 737-800 (winglets),BER
4,TP 530,TAP Air Portugal,Lisbon,LIS,2026-05-29 12:20+02:00,Expected,Airbus A320-200 (sharklets),BER
...,...,...,...,...,...,...,...,...
196,FR 3361,Ryanair,Palermo,PMO,2026-05-29 23:00+02:00,Expected,Boeing 737-800 (winglets),BER
197,EW 8231,Eurowings,Helsinki,HEL,2026-05-29 23:00+02:00,Expected,Airbus A320 NEO,BER
198,EW 8671,Eurowings,Kos Island,KGS,2026-05-29 23:05+02:00,Expected,Airbus A321 NEO,BER
199,SR 7359,SundAir,Burgas,BOJ,2026-05-29 23:05+02:00,Expected,Airbus A320,BER


In [ ]:
# Function to retrieve flight data for multiple airports and inspect results
import requests
import pandas as pd

def flight_arrivals_dataframe(airport_codes):

    all_arrivals = []

    headers = {
        "x-rapidapi-key": os.getenv("RAPID_API_KEY"),
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    for code in airport_codes:

        url = f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/{code}/2026-05-29T12:00/2026-05-29T23:59"

        querystring = {
            "withLeg": "true",
            "direction": "Both",
            "withCancelled": "false",
            "withCodeshared": "false",
            "withCargo": "false",
            "withPrivate": "false",
            "withLocation": "false"
        }

        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code != 200:
            print(f"Error for {code}: {response.status_code}")
            print(response.text)   # important for debugging
            continue

        data = response.json()

        for flight in data.get("arrivals", []):

            all_arrivals.append({
                "airport": code,
                "flight_number": flight.get("number"),
                "airline": flight.get("airline", {}).get("name"),
                "departure_airport": flight.get("departure", {}).get("airport", {}).get("name"),
                "departure_iata": flight.get("departure", {}).get("airport", {}).get("iata"),
                "arrival_time_local": flight.get("arrival", {}).get("scheduledTime", {}).get("local"),
                "status": flight.get("status"),
                "aircraft": flight.get("aircraft", {}).get("model")
            })

    return pd.DataFrame(all_arrivals)

In [47]:
# Use function to retrieve flight data for multiple airports and inspect results
airports = ["MUC", "HAM"]

flights_df = flight_arrivals_dataframe(airports)

flights_df.head()

Error for HAM: 429
{"message":"You have exceeded the rate limit per second for your plan, BASIC, by the API provider"}


,airport,flight_number,airline,departure_airport,departure_iata,arrival_time_local,status,aircraft
0,MUC,LH 1935,Lufthansa,Berlin,BER,2026-05-29 12:00+02:00,Expected,Airbus A320
1,MUC,GQ 870,Sky Express,Athens,ATH,2026-05-29 12:00+02:00,Expected,Airbus A320 NEO
2,MUC,4Y 441,Eurowings Discover,Ibiza Town,IBZ,2026-05-29 12:05+02:00,Expected,Airbus A320
3,MUC,LH 2055,Lufthansa,Hamburg,HAM,2026-05-29 12:05+02:00,Expected,Airbus A321
4,MUC,DE 1469,Condor,Heraklion,HER,2026-05-29 12:15+02:00,Expected,Airbus A320


In [48]:
flights_df

,airport,flight_number,airline,departure_airport,departure_iata,arrival_time_local,status,aircraft
0,MUC,LH 1935,Lufthansa,Berlin,BER,2026-05-29 12:00+02:00,Expected,Airbus A320
1,MUC,GQ 870,Sky Express,Athens,ATH,2026-05-29 12:00+02:00,Expected,Airbus A320 NEO
2,MUC,4Y 441,Eurowings Discover,Ibiza Town,IBZ,2026-05-29 12:05+02:00,Expected,Airbus A320
3,MUC,LH 2055,Lufthansa,Hamburg,HAM,2026-05-29 12:05+02:00,Expected,Airbus A321
4,MUC,DE 1469,Condor,Heraklion,HER,2026-05-29 12:15+02:00,Expected,Airbus A320
...,...,...,...,...,...,...,...,...
270,MUC,DE 1523,Condor,Gran Canaria Island,LPA,2026-05-29 23:15+02:00,Expected,Airbus A320
271,MUC,TP 556,TAP Air Portugal,Lisbon,LIS,2026-05-29 23:20+02:00,Expected,Airbus A320
272,MUC,LH 2481,Lufthansa,London,LHR,2026-05-29 23:20+02:00,Expected,Airbus A320 NEO
273,MUC,4Y 453,Eurowings Discover,Palma De Mallorca,PMI,2026-05-29 23:20+02:00,Expected,Airbus A320


In [51]:
# Put together BER and MUC arrival data
all_flights = pd.concat([arrivals_df, flights_df], ignore_index=True)

In [52]:
all_flights

,flight_number,airline,departure_airport,departure_iata,arrival_time_local,status,aircraft,airport
0,LH 1926,Lufthansa,Munich,MUC,2026-05-29 12:05+02:00,Expected,Airbus A321-200,BER
1,U2 5004,easyJet,Kerkyra Island,CFU,2026-05-29 12:05+02:00,Expected,Airbus A321 NEO,BER
2,FR 227,Ryanair,Palma De Mallorca,PMI,2026-05-29 12:10+02:00,Expected,Boeing 737 MAX 8,BER
3,FR 5475,Ryanair,Marrakech,RAK,2026-05-29 12:20+02:00,Expected,Boeing 737-800 (winglets),BER
4,TP 530,TAP Air Portugal,Lisbon,LIS,2026-05-29 12:20+02:00,Expected,Airbus A320-200 (sharklets),BER
...,...,...,...,...,...,...,...,...
471,DE 1523,Condor,Gran Canaria Island,LPA,2026-05-29 23:15+02:00,Expected,Airbus A320,MUC
472,TP 556,TAP Air Portugal,Lisbon,LIS,2026-05-29 23:20+02:00,Expected,Airbus A320,MUC
473,LH 2481,Lufthansa,London,LHR,2026-05-29 23:20+02:00,Expected,Airbus A320 NEO,MUC
474,4Y 453,Eurowings Discover,Palma De Mallorca,PMI,2026-05-29 23:20+02:00,Expected,Airbus A320,MUC


In [ ]:
# Save data to CSV
arrivals_df.to_csv("berlin_flights.csv", index=False)
flights_df.to_csv("munich_flights.csv", index=False)

In [57]:
# Send to SQL
all_flights.to_sql(
    "flights",
    con=engine,
    if_exists="append",
    index=False
)

476